# Chapter 01 — Beyond the Chat Box

**Companion to *Applied AI*.**

This notebook accompanies Chapter 1, **Beyond the Chat Box**. The chapter makes
the argument; this notebook lets you touch the mechanism.

## Question

A paragraph review produced in a chat window is text in a scroll buffer.
**What can software do with that review once it becomes a value bound to the
input it judged?**

## What this notebook establishes

- A model call that returns a bare string cannot answer *which paragraph was
  this about*, *which rules judged it*, or *has it been dealt with*.
- Three stored fields make those three questions answerable by a program.
- Once a review is bound to a content hash, a program can **refuse** to file a
  review whose paragraph has since changed. That refusal is the chapter's
  "task identity in its smallest form".
- Keeping failure in its own channel prevents the most common integration
  defect: filing a diagnostic string as though it were a review.

## What this notebook does **not** establish

- Nothing here calls a real model. The adapter is a local fake with a canned
  response, so **no claim is made about model ability, review quality, or
  factual verification**.
- Storing a review next to a hash does not make the review correct. Chapter 21
  is where verification is built.
- This is a **demonstration** of an architectural boundary, not a measurement.

## Setup

Standard library only. No network, no API key, no `codeai` import.

In [1]:
import hashlib
import json
from dataclasses import dataclass, field
from typing import Optional

PROMPT_VERSION = "claims-review-v1"

PARAGRAPH = "The service processes every request within one second."

INSTRUCTION = (
    "Identify factual claims that require evidence. "
    "Do not invent sources. Return a short review."
)

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("paragraph:", PARAGRAPH)
print("sha256    :", sha256(PARAGRAPH)[:16], "...")

paragraph: The service processes every request within one second.
sha256    : df21c85a4a273157 ...


## A fake model adapter

The chapter's `OpenCodeCognitionAdapter` returns a `CallResult` carrying the
output, the usage, and the failure **in its own status channel**. The fake
below keeps that shape and nothing else. It can be told to fail, so we can see
what a caller does with a failure.

In [2]:
@dataclass(frozen=True)
class CallResult:
    """Minimal stand-in for the chapter's CallResult."""
    status: str                      # "succeeded" | "failed"
    raw_output: Optional[str] = None # content channel
    error: Optional[str] = None      # failure channel, kept separate
    usage: dict = field(default_factory=dict)

CANNED_REVIEW = (
    'The sentence "processes every request within one second" is a '
    "measurable claim and cites no measurement. It needs a source."
)

class FakeAdapter:
    """Deterministic local stand-in for a model. No network."""

    def __init__(self, fail_with: Optional[str] = None):
        self.fail_with = fail_with
        self.calls = 0

    def prompt(self, text: str, instruction: str = "") -> CallResult:
        self.calls += 1
        if self.fail_with:
            return CallResult(status="failed", error=self.fail_with)
        return CallResult(
            status="succeeded",
            raw_output=CANNED_REVIEW,
            usage={"input_tokens": 38, "output_tokens": 26, "source": "fake"},
        )

adapter = FakeAdapter()
result = adapter.prompt(PARAGRAPH, instruction=INSTRUCTION)
print(result.status)
print(result.raw_output[:70], "...")

succeeded
The sentence "processes every request within one second" is a measurab ...


## Baseline: the chat arrangement

In chat, the reviewer is a string. Here it is, and here is what a program can
ask it.

In [3]:
review_text = result.raw_output   # this is all a chat window leaves you

questions = [
    "Which paragraph was this about?",
    "Which version of that paragraph?",
    "Which rules judged it?",
    "Has it been dealt with?",
]

print("A bare string can answer:")
for q in questions:
    print(f"  {q:38s} -> no")
print()
print(f"len(review_text) = {len(review_text)}  # the only property available")

A bare string can answer:
  Which paragraph was this about?        -> no
  Which version of that paragraph?       -> no
  Which rules judged it?                 -> no
  Has it been dealt with?                -> no

len(review_text) = 123  # the only property available


The string is not *wrong*. It is simply not connected to anything. In the
chapter's terms, the operator is still holding task identity and state in their
head, where it cannot be queried, tested, or resumed.

## Experiment: move three operator jobs into stored fields

The chapter names seven jobs the operator does invisibly and moves three of
them. Each becomes one field.

| Operator job | Stored field |
|---|---|
| Chose the paragraph | `paragraph_id`, `paragraph_sha` |
| Supplied the criteria | `prompt_version` |
| Moved the result back | `disposition` |

In [4]:
def review_paragraph(paragraph_id: str, paragraph: str, adapter) -> dict:
    """Produce a review record, or a failure record. Never mix the two."""
    res = adapter.prompt(paragraph, instruction=INSTRUCTION)

    if res.status != "succeeded":
        # The failure never enters the content channel.
        return {
            "paragraph_id": paragraph_id,
            "paragraph_sha": sha256(paragraph),
            "prompt_version": PROMPT_VERSION,
            "status": "failed",
            "error": res.error,
            "review": None,
            "disposition": "unavailable",
        }

    return {
        "paragraph_id": paragraph_id,
        "paragraph_sha": sha256(paragraph),
        "prompt_version": PROMPT_VERSION,
        "model": "fake-model",
        "status": "succeeded",
        "error": None,
        "review": res.raw_output,      # content channel only
        "disposition": "pending",
    }

record = review_paragraph("ch3-para-014", PARAGRAPH, FakeAdapter())
print(json.dumps({k: (v[:48] + "..." if isinstance(v, str) and len(v) > 48 else v)
                  for k, v in record.items()}, indent=2))

{
  "paragraph_id": "ch3-para-014",
  "paragraph_sha": "df21c85a4a273157de7e17f73396b649742ab9b7fbb77402...",
  "prompt_version": "claims-review-v1",
  "model": "fake-model",
  "status": "succeeded",
  "error": null,
  "review": "The sentence \"processes every request within one...",
  "disposition": "pending"
}


## Observation

The same four questions, asked of the record rather than of the string.

In [5]:
def answer(record, paragraph_now: str):
    return {
        "Which paragraph was this about?": record["paragraph_id"],
        "Which version of that paragraph?": record["paragraph_sha"][:12] + "...",
        "Which rules judged it?": record["prompt_version"],
        "Has it been dealt with?": record["disposition"],
        "Is the review still about the current text?":
            record["paragraph_sha"] == sha256(paragraph_now),
    }

for q, a in answer(record, PARAGRAPH).items():
    print(f"{q:46s} -> {a}")

Which paragraph was this about?                -> ch3-para-014
Which version of that paragraph?               -> df21c85a4a27...
Which rules judged it?                         -> claims-review-v1
Has it been dealt with?                        -> pending
Is the review still about the current text?    -> True


## The one new thing the reader can now do

Hash any paragraph, attach the review to that hash rather than to memory, and
**refuse to file a review whose `paragraph_sha` no longer matches the file**.

In [6]:
class StaleReview(Exception):
    pass

def file_review(record: dict, paragraph_now: str) -> dict:
    """Move a pending review into the manuscript, or refuse."""
    if record["status"] != "succeeded":
        raise StaleReview(f"cannot file a failed review: {record['error']}")
    current = sha256(paragraph_now)
    if record["paragraph_sha"] != current:
        raise StaleReview(
            "paragraph changed since review: "
            f"reviewed {record['paragraph_sha'][:12]}..., "
            f"file now {current[:12]}..."
        )
    filed = dict(record)
    filed["disposition"] = "filed"
    return filed

filed = file_review(record, PARAGRAPH)
print("filed  ->", filed["disposition"])

filed  -> filed


## Failure: the paragraph moves under the review

This is the stale-review defect from the chapter: a delayed response attached
to an edited paragraph. In chat nothing notices. Here it is refused.

In [7]:
edited = "The service processes most requests within one second."

print("original:", PARAGRAPH)
print("edited  :", edited)
print()

try:
    file_review(record, edited)
except StaleReview as exc:
    print("REFUSED:", exc)

original: The service processes every request within one second.
edited  : The service processes most requests within one second.

REFUSED: paragraph changed since review: reviewed df21c85a4a27..., file now 0f3adf7c5d40...


Note what was *not* required to catch this: no model call, no cleverness, and
no reviewer paying attention. A hash comparison did it, which is Chapter 3's
determinism rule arriving early.

## Failure: reading an error as content

The chapter calls this "the single most common integration defect in the wild,
and the cheapest to fix". Below, the same pipeline is run against an adapter
that fails. Compare the careless path with the recorded one.

In [8]:
broken = FakeAdapter(fail_with="HTTP 403: edge block, credential not evaluated")

# The careless path: whatever came back becomes the review.
careless = broken.prompt(PARAGRAPH, instruction=INSTRUCTION)
careless_review = careless.raw_output or careless.error   # <- the defect
print("careless review field:", repr(careless_review)[:70])

# The recorded path: failure stays in its own channel.
failed_record = review_paragraph("ch3-para-014", PARAGRAPH, broken)
print()
print("recorded review field :", failed_record["review"])
print("recorded error field  :", failed_record["error"])
print("recorded disposition  :", failed_record["disposition"])

assert failed_record["review"] is None, "a failure must never enter the content channel"
print()
print("assertion held: the diagnostic string never became a review")

careless review field: 'HTTP 403: edge block, credential not evaluated'

recorded review field : None
recorded error field  : HTTP 403: edge block, credential not evaluated
recorded disposition  : unavailable

assertion held: the diagnostic string never became a review


## Interpretation

Three things are now true that were not true of the string.

1. **The review is about something.** `paragraph_id` and `paragraph_sha` name
   the exact bytes that were judged. That is task identity in its smallest
   form.
2. **The review was judged by something.** `prompt_version` records which rules
   applied, so two reviews produced under different rules are not silently
   compared.
3. **The review has a state.** `disposition` distinguishes work that is pending
   from work that is finished, so it cannot be done twice or never.

What has **not** changed: the model is no smarter, and nothing has been
verified. A generated review is a proposal about where to look. Transport
correctness establishes nothing about whether the service meets its latency
promise.

One responsibility moved out of human memory and into the process. That is
Applied AI at its smallest size.

## Try it yourself

1. **Add a fourth field.** The chapter lists seven operator jobs; three are
   stored here. Add `reviewed_at` or `reviewer_id` and decide what failure each
   one catches. If you cannot name the failure, the field is decoration.
2. **Break the binding deliberately.** Change `file_review` to compare
   `paragraph_id` instead of `paragraph_sha`, then re-run the stale-paragraph
   cell. Watch the edited paragraph accept a review it never received.
3. **Make the adapter non-deterministic.** Return one of two canned reviews at
   random and call it twice on the same paragraph. Which fields differ, and
   which must not? This is the question Chapter 10 turns into a release
   protocol.
4. **Try to file twice.** What should `file_review` do when `disposition` is
   already `"filed"`? Chapter 22 answers this properly; write down your answer
   first.